In [7]:
import csv
import os
import random
import nest_asyncio
nest_asyncio.apply()

os.makedirs("data", exist_ok=True)

feedback_items = [
    ("support_ticket", "Enterprise", "App crashed again while exporting, this is the second time this week."),
    ("support_ticket", "Pro", "App crashed while exporting a large report."),
    ("app_review", "Free", "Export feature crashed my whole session, lost my work."),
    ("survey", "Enterprise", "Exporting to CSV fails silently, no error message at all."),
    ("support_ticket", "Pro", "Export button does nothing on Safari."),
    ("app_review", "Free", "Please add dark mode, my eyes are dying using this at night."),
    ("app_review", "Pro", "Would love a dark mode option, half our team works late."),
    ("survey", "Free", "Dark mode please! Asked this three months ago."),
    ("app_review", "Enterprise", "Can you add integration with Slack? We use it for everything."),
    ("support_ticket", "Enterprise", "Slack integration would save us so much manual copy pasting."),
    ("survey", "Pro", "Slack or Teams integration would be great."),
    ("support_ticket", "Enterprise", "Customer support took 3 days to respond to my billing question."),
    ("survey", "Pro", "Billing dashboard shows the wrong renewal date, very confusing."),
    ("app_review", "Free", "Pricing page is confusing, I couldn't tell which plan I was on."),
    ("support_ticket", "Enterprise", "Was overcharged this month, invoice does not match my plan."),
    ("survey", "Pro", "The new dashboard is so much faster than before, great job!"),
    ("app_review", "Free", "Your support team was amazing, fixed my issue in 10 minutes."),
    ("survey", "Enterprise", "Really happy with the recent speed improvements on load times."),
    ("support_ticket", "Pro", "Search inside the app is broken, returns no results for common terms."),
    ("app_review", "Enterprise", "Search feature never returns relevant results, very frustrating."),
    ("survey", "Free", "Search is basically unusable for our team."),
    ("support_ticket", "Pro", "Mobile app logs me out randomly every few hours."),
    ("app_review", "Enterprise", "Getting logged out constantly on mobile, very disruptive."),
    ("survey", "Free", "Would like a way to bulk delete old reports."),
    ("app_review", "Pro", "Bulk actions would save us hours every month."),
    ("support_ticket", "Enterprise", "API rate limits are too low for our integration."),
    ("survey", "Enterprise", "Need higher API rate limits, we are hitting them daily."),
    ("app_review", "Free", "Onboarding was confusing, took me a while to find key features."),
    ("survey", "Pro", "Great onboarding experience, very smooth setup."),
    ("support_ticket", "Free", "Notification emails are being marked as spam."),
]

rows = [["ticket_id", "date", "source", "customer_tier", "feedback_text"]]
for i, (source, tier, text) in enumerate(feedback_items, start=1001):
    rows.append([f"T-{i}", f"2025-06-{(i % 28) + 1:02d}", source, tier, text])

with open("data/feedback.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(f"Wrote {len(rows) - 1} feedback rows to data/feedback.csv")

Wrote 30 feedback rows to data/feedback.csv


# Task 1: Multi-Agent Design Thinking

## Business Task Chosen

**Customer Feedback to Product Prioritization**

Given a batch of raw customer feedback (support tickets, app reviews, survey
responses), the crew reads through it, categorizes and merges duplicate
issues, scores and ranks them by frequency, severity, and customer tier,
then produces a plain language stakeholder report the product team can act
on directly.

## The Three Agent Roles

### Agent 1: Feedback Aggregator
- **Role:** Customer Feedback Analyst
- **Goal:** Extract and categorize all distinct feature requests, bugs, and
  complaints from raw customer feedback, tracking which customer tier each
  mention came from
- **Backstory:** Has spent years reading through messy support tickets and
  reviews, skilled at spotting the real signal in rambling customer text and
  organizing it into clean, consistent categories while preserving metadata
  like customer tier for later analysis

### Agent 2: Prioritizer
- **Role:** Product Prioritization Strategist
- **Goal:** Score and rank feedback items using a consistent, auditable
  weighting formula that accounts for frequency, severity, and customer
  tier
- **Backstory:** A former product manager who has shipped roadmaps for
  years, never ranks by gut feeling alone, applies a consistent scoring
  formula every time so the prioritization can be checked and trusted by
  anyone on the team

### Agent 3: Report Writer
- **Role:** Stakeholder Communications Writer
- **Goal:** Turn the ranked feedback into a clear, executive-ready summary
- **Backstory:** Translates dense analysis into short, confident business
  language, knows executives want the so-what up front rather than a data
  dump

## Why Specialized Agents Over One Generalist

Splitting the work keeps each agent's prompt focused on exactly one skill,
extraction, scoring, or writing, which avoids a single agent blurring
stages together (for example scoring before extraction is finished). It
also makes each handoff auditable, you can inspect Agent 1's categorized
output alone before Agent 2 ever touches it. This is not automatically
better though, for a small dataset a single well-prompted agent could
likely produce similar quality faster and cheaper, the benefit of
specialization grows with data volume, scoring complexity, and the need to
audit or swap stages independently.

In [23]:
import os
from dotenv import load_dotenv, find_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import FileReadTool, FileWriterTool

load_dotenv(find_dotenv())

BASE_URL = "https://llm.netixsol.com/v1"
MODEL_NAME = "openai/coder"
GATEWAY_API_KEY = os.getenv("GATEWAY_API_KEY")

llm_aggregator = LLM(
    model=MODEL_NAME,
    base_url=BASE_URL,
    api_key=GATEWAY_API_KEY,
    temperature=0.1,
)

llm_prioritizer = LLM(
    model=MODEL_NAME,
    base_url=BASE_URL,
    api_key=GATEWAY_API_KEY,
    temperature=0.2,
)

llm_writer = LLM(
    model=MODEL_NAME,
    base_url=BASE_URL,
    api_key=GATEWAY_API_KEY,
    temperature=0.4,
)

print("LLM clients ready, pointed at:", BASE_URL)

LLM clients ready, pointed at: https://llm.netixsol.com/v1


## Tools

In [3]:
file_read_tool = FileReadTool(file_path="data/feedback.csv")
file_write_tool = FileWriterTool()

# Task 2: Build Agents & Assign Tools

## Agent to Tool Mapping

| Agent | Tools | Justification |
|---|---|---|
| Feedback Aggregator | FileReadTool | Only job is ingesting raw data from the feedback CSV, no reason to write files or score anything |
| Prioritizer | None | Ranking is a judgment and arithmetic task on data it already receives via context, giving it tools here would be bloat with no real use |
| Report Writer | FileWriterTool | Needs to persist the final report to disk, does not need raw data access or scoring, both are done upstream |

## Design Principle

Each agent has exactly the tools its responsibility requires and nothing
more. This mirrors real team permissioning, an analyst reading customer
data does not need write access to the final report, and a writer does
not need raw data access when working from an already-processed input.

Each agent also has its own `LLM` instance so temperature can be tuned per
role, lower temperature for the Aggregator and Prioritizer since they need
consistency and correctness, higher temperature for the Report Writer
since it benefits from more natural, varied language.

In [4]:
feedback_aggregator = Agent(
    role="Customer Feedback Analyst",
    goal="Extract and categorize all distinct feature requests, bugs, and complaints from raw customer feedback, tracking which customer tier each mention came from",
    backstory=(
        "You have spent years reading through messy support tickets and reviews. "
        "You are skilled at spotting the real signal in rambling customer text "
        "and organizing it into clean, consistent categories, while carefully "
        "preserving metadata like customer tier for later analysis."
    ),
    tools=[file_read_tool],
    llm=llm_aggregator,
    verbose=True,
)

prioritizer = Agent(
    role="Product Prioritization Strategist",
    goal="Score and rank feedback items using a consistent, auditable weighting formula that accounts for frequency, severity, and customer tier",
    backstory=(
        "You are a former product manager who has shipped roadmaps for years. "
        "You never rank by gut feeling alone. You apply a consistent scoring "
        "formula every time so your prioritization can be checked and trusted "
        "by anyone on the team."
    ),
    tools=[],
    llm=llm_prioritizer,
    verbose=True,
)

report_writer = Agent(
    role="Stakeholder Communications Writer",
    goal="Turn the ranked feedback into a clear, executive-ready summary",
    backstory=(
        "You translate dense analysis into short, confident business language. "
        "You know executives want the so-what up front, not a data dump."
    ),
    tools=[file_write_tool],
    llm=llm_writer,
    verbose=True,
)

# Task 3: Define Tasks & Process

## Task Chain

Three `Task` objects, one per agent, wired together with `context`:

- `task1_extract` runs first, no dependencies, reads the raw CSV
- `task2_prioritize` depends on `task1_extract` via `context=[task1_extract]`
- `task3_report` depends on `task2_prioritize` via `context=[task2_prioritize]`

This forms a strict pipeline: extraction output feeds scoring, scoring
output feeds the final report. Assembled into a `Crew` and run with
`Process.sequential`, meaning tasks execute strictly in the order listed,
each one waiting for the previous to finish.


In [5]:
task1_extract = Task(
    description=(
        "Read the customer feedback CSV file at data/feedback.csv. Extract "
        "every distinct issue mentioned across all rows. Categorize each "
        "issue into exactly one of: Feature Request, Bug, Complaint, or "
        "Praise. Merge mentions of the same underlying issue together into "
        "one row, even if worded differently. For each merged item, list "
        "the customer_tier of every person who mentioned it."
    ),
    expected_output=(
        "A markdown table with exactly these columns: Category, Item, "
        "Mentions, Tiers. The Tiers column should be a comma-separated list "
        "of tier names, one per mention, for example 'Enterprise, Pro, Pro'. "
        "One row per distinct issue. No prose before or after the table."
    ),
    agent=feedback_aggregator,
)

task2_prioritize = Task(
    description=(
        "Using the categorized feedback table, calculate a Priority Score "
        "for each item using this exact formula:\n"
        "1. Base score equals number of mentions multiplied by 2\n"
        "2. Add severity points based on category: Bug = plus 5, "
        "Complaint = plus 3, Feature Request = plus 2, Praise = plus 0\n"
        "3. Add tier weighting, counted per mention: Enterprise = plus 3, "
        "Pro = plus 1, Free = plus 0\n"
        "4. Sum all three parts into a final Priority Score per item\n"
        "Show the tier breakdown for each item so the weighting is auditable."
    ),
    expected_output=(
        "A markdown table with exactly these columns: Rank, Item, Category, "
        "Mentions, Tier Breakdown, Priority Score. Tier Breakdown should "
        "read like '2 Enterprise, 1 Pro' so the source of the weight is "
        "clear. Sorted by Priority Score descending. No prose before or "
        "after the table."
    ),
    agent=prioritizer,
    context=[task1_extract],
)

task3_report = Task(
    description=(
        "Using the ranked priority table, write a one-page stakeholder "
        "report for the product team. Lead with the top 3 priorities and "
        "explain why each matters, referencing frequency and which customer "
        "tiers are affected. Then briefly summarize the rest of the list. "
        "Save the final report to data/report.md using the file writer tool."
    ),
    expected_output=(
        "A markdown document with a short intro paragraph, a Top Priorities "
        "section covering the top 3 items with a one to two sentence "
        "rationale each, and a Full Ranking section listing everything else "
        "briefly. Written in plain business language, no raw score tables."
    ),
    agent=report_writer,
    context=[task2_prioritize],
)

# Sequential Crew

In [9]:
crew_sequential = Crew(
    agents=[feedback_aggregator, prioritizer, report_writer],
    tasks=[task1_extract, task2_prioritize, task3_report],
    process=Process.sequential,
    verbose=True,
)

result_sequential = await crew_sequential.kickoff_async()

print("\n\n===== SEQUENTIAL RESULT =====\n")
print(result_sequential)
print("\n===== SEQUENTIAL TOKEN USAGE =====\n")
print(crew_sequential.usage_metrics)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b8170d2e-67ab-4ed4-ba59-00ededaa8a54                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read the customer feedback CSV file at data/feedback.csv. Extract every distinct issue mentioned across  │
│  all rows. Categorize each issue into exactly one of: Feature Request, Bug, Complaint, or Praise. Merge         │
│  mentions of the same underlying issue together into one row, even if worded differently. For each merged       │
│  item, list the customer_tier of every person who mentioned it.                                                 │
│  ID: 70f8ede9-85dd-4c64-9880-c5fc55d90945                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Feedback Analyst                                                                               │
│                                                                                                                 │
│  Task: Read the customer feedback CSV file at data/feedback.csv. Extract every distinct issue mentioned across  │
│  all rows. Categorize each issue into exactly one of: Feature Request, Bug, Complaint, or Praise. Merge         │
│  mentions of the same underlying issue together into one row, even if worded differently. For each merged       │
│  item, list the customer_tier of every person who mentioned it.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Args: {'start_line': 1, 'line_count': 5, 'file_path': 'data/feedback.csv'}                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_a_files_content executed with result: ticket_id,date,source,customer_tier,feedback_text
T-1001,2025-06-22,support_ticket,Enterprise,"App crashed again while exporting, this is the second time this week."
T-1002,2025-06-23,support_ticket,P...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stakeholder Communications Writer                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  This report summarizes recent product feedback, highlighting key areas for the product team's attention. Our   │
│  analysis prioritizes issues and requests based on frequency and impact across customer tiers, ensuring we      │
│  focus on improvements that will deliver the most value to our users.                                           │
│                                                                                                                 │
│  ## Top Priorities                                                                                              │
│                                                                                                                 │
│  1.  **App crashes during export:** This critical bug, reported by Enterprise, Pro, and Free users, indicates   │
│  a fundamental instability in a core product function. Addressing this will prevent significant user            │
│  frustration and data loss across all customer segments.                                                        │
│  2.  **Slack/Teams integration feature request:** Highly requested by our Enterprise and Pro customers, this    │
│  integration is vital for enhancing workflow efficiency and collaboration for our most valuable business        │
│  users. Implementing this will significantly improve the product's utility within professional environments.    │
│  3.  **Search feature issues (broken/irrelevant results/unusable):** Affecting all tiers (Enterprise, Pro, and  │
│  Free), the current search functionality is a major impediment to user productivity. Resolving these issues     │
│  will improve basic usability and help users quickly find the information they need.                            │
│                                                                                                                 │
│  ## Full Ranking                                                                                                │
│                                                                                                                 │
│  Beyond the top three, several other items warrant attention:                                                   │
│                                                                                                                 │
│  *   **Random logouts on mobile app:** A bug impacting Enterprise and Pro users, causing disruption.            │
│  *   **Low API rate limits:** A complaint from Enterprise users, suggesting a bottleneck for heavy usage.       │
│  *   **Export to CSV fails silently:** A critical bug affecting Enterprise users, leading to data integrity     │
│  concerns.                                                                                                      │
│  *   **Dark mode feature request:** A popular aesthetic request, particularly from Pro and Free users.          │
│  *   **Export button not working on Safari:** A browser-specific bug affecting Pro users.                       │
│  *   **Slow customer support response for billing:** A complaint from Enterprise users, indicating a need for   │
│  improved billing support.                                                                                      │
│  *   **Incorrect renewal date in billing dashboard:** A bug affecting Pro users, leading to potential           │
│  confusion.                                            

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the ranked priority table, write a one-page stakeholder report for the product team. Lead with     │
│  the top 3 priorities and explain why each matters, referencing frequency and which customer tiers are          │
│  affected. Then briefly summarize the rest of the list. Save the final report to data/report.md using the file  │
│  writer tool.                                                                                                   │
│  Agent: Stakeholder Communications Writer                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



===== SEQUENTIAL RESULT =====

This report summarizes recent product feedback, highlighting key areas for the product team's attention. Our analysis prioritizes issues and requests based on frequency and impact across customer tiers, ensuring we focus on improvements that will deliver the most value to our users.

## Top Priorities

1.  **App crashes during export:** This critical bug, reported by Enterprise, Pro, and Free users, indicates a fundamental instability in a core product function. Addressing this will prevent significant user frustration and data loss across all customer segments.
2.  **Slack/Teams integration feature request:** Highly requested by our Enterprise and Pro customers, this integration is vital for enhancing workflow efficiency and collaboration for our most valuable business users. Implementing this will significantly improve the product's utility within professional environments.
3.  **Search feature issues (broken/irrelevant results/unusable):** Affecting 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b8170d2e-67ab-4ed4-ba59-00ededaa8a54                                                                       │
│  Final Output: This report summarizes recent product feedback, highlighting key areas for the product team's    │
│  attention. Our analysis prioritizes issues and requests based on frequency and impact across customer tiers,   │
│  ensuring we focus on improvements that will deliver the most value to our users.                               │
│                                                                                                                 │
│  ## Top Priorities                                                                                              │
│                                                                                                                 │
│  1.  **App crashes during export:** This critical bug, reported by Enterprise, Pro, and Free users, indicates   │
│  a fundamental instability in a core product function. Addressing this will prevent significant user            │
│  frustration and data loss across all customer segments.                                                        │
│  2.  **Slack/Teams integration feature request:** Highly requested by our Enterprise and Pro customers, this    │
│  integration is vital for enhancing workflow efficiency and collaboration for our most valuable business        │
│  users. Implementing this will significantly improve the product's utility within professional environments.    │
│  3.  **Search feature issues (broken/irrelevant results/unusable):** Affecting all tiers (Enterprise, Pro, and  │
│  Free), the current search functionality is a major impediment to user productivity. Resolving these issues     │
│  will improve basic usability and help users quickly find the information they need.                            │
│                                                                                                                 │
│  ## Full Ranking                                                                                                │
│                                                                                                                 │
│  Beyond the top three, several other items warrant attention:                                                   │
│                                                                                                                 │
│  *   **Random logouts on mobile app:** A bug impacting Enterprise and Pro users, causing disruption.            │
│  *   **Low API rate limits:** A complaint from Enterprise users, suggesting a bottleneck for heavy usage.       │
│  *   **Export to CSV fails silently:** A critical bug affecting Enterprise users, leading to data integrity     │
│  concerns.                                                                                                      │
│  *   **Dark mode feature request:** A popular aesthetic request, particularly from Pro and Free users.          │
│  *   **Export button not working on Safari:** A browser-specific bug affecting Pro users.                       │
│  *   **Slow customer support response for billing:** A complaint from Enterprise users, indicating a need for   │
│  improved billing support.                                                                                      │
│  *   **Incorrect renewal date in billing dashboard:** A bug affecting Pro users, leading to potential           │
│  confusion.                                           

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#  Task 4: Hierarchical Crew

In [11]:
manager = Agent(
    role="Product Insights Manager",
    goal=(
        "Ensure the final stakeholder report is accurate, correctly "
        "prioritized using the agreed formula, and ready to hand to "
        "leadership without further edits"
    ),
    backstory=(
        "You oversee the feedback to roadmap pipeline. You review each "
        "stage of work before it moves forward, check that the "
        "prioritization formula was applied correctly, and send work back "
        "for revision if it does not meet the bar."
    ),
    llm=llm_prioritizer,
    allow_delegation=True,
    verbose=True,
)

crew_hierarchical = Crew(
    agents=[feedback_aggregator, prioritizer, report_writer],
    tasks=[task1_extract, task2_prioritize, task3_report],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
)

result_hierarchical = await crew_hierarchical.kickoff_async()

print("\n\n===== HIERARCHICAL RESULT =====\n")
print(result_hierarchical)
print("\n===== HIERARCHICAL TOKEN USAGE =====\n")
print(crew_hierarchical.usage_metrics)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3e9972aa-e590-4a3a-8055-29b3eb319301                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read the customer feedback CSV file at data/feedback.csv. Extract every distinct issue mentioned across  │
│  all rows. Categorize each issue into exactly one of: Feature Request, Bug, Complaint, or Praise. Merge         │
│  mentions of the same underlying issue together into one row, even if worded differently. For each merged       │
│  item, list the customer_tier of every person who mentioned it.                                                 │
│  ID: 70f8ede9-85dd-4c64-9880-c5fc55d90945                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Product Insights Manager                                                                                │
│                                                                                                                 │
│  Task: Read the customer feedback CSV file at data/feedback.csv. Extract every distinct issue mentioned across  │
│  all rows. Categorize each issue into exactly one of: Feature Request, Bug, Complaint, or Praise. Merge         │
│  mentions of the same underlying issue together into one row, even if worded differently. For each merged       │
│  item, list the customer_tier of every person who mentioned it.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Args: {'file_path': 'data/feedback.csv'}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_a_files_content executed with result: ticket_id,date,source,customer_tier,feedback_text
T-1001,2025-06-22,support_ticket,Enterprise,"App crashed again while exporting, this is the second time this week."
T-1002,2025-06-23,support_ticket,P...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Product Insights Manager                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Product Feedback Stakeholder Report                                                                          │
│                                                                                                                 │
│  This report summarizes recent product feedback, highlighting key priorities for the product team based on      │
│  frequency of mentions and impact across customer tiers. The insights gathered will help inform our product     │
│  roadmap and ensure we are addressing the most critical needs and opportunities.                                │
│                                                                                                                 │
│  ## Top Priorities                                                                                              │
│                                                                                                                 │
│  1.  **Export Feature Issues:** This is the most frequently reported issue with 5 mentions, significantly       │
│  impacting Enterprise and Pro users, as well as Free users. Addressing this bug is crucial for core             │
│  functionality and user satisfaction.                                                                           │
│                                                                                                                 │
│  2.  **Billing/Pricing/Support Issues:** With 4 mentions, these complaints are critical as they directly        │
│  affect customer satisfaction and retention, particularly for our Enterprise clients, but also Pro and Free     │
│  users.                                                                                                         │
│                                                                                                                 │
│  3.  **Search Feature Issues:** This bug, mentioned 3 times, affects a fundamental aspect of the product and    │
│  is reported across all customer tiers (Enterprise, Pro, and Free), indicating a broad impact on usability.     │
│                                                                                                                 │
│  ## Full Ranking                                                                                                │
│                                                                                                                 │
│  Below is a summary of the remaining feedback items:                                                            │
│                                                                                                                 │
│  *   **4. Slack/Teams Integration:** A highly requested feature (3 mentions) primarily from our Enterprise and  │
│  Pro users.                                                                                                     │
│  *   **5. Mobile App Logout Bug:** An important bug (2 mentions) affecting the mobile experience for            │
│  Enterprise and Pro users.                                                                                      │
│  *   **6. Higher API Rate Limits:** A specific feature request (2 mentions) from our Enterprise customers.      │
│  *   **7. Dark Mode:** A popular feature request (3 mentions) with interest from Pro and Free users.            │
│  *   **8. Speed Improvements:** Positive feedback (2 me

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the ranked priority table, write a one-page stakeholder report for the product team. Lead with     │
│  the top 3 priorities and explain why each matters, referencing frequency and which customer tiers are          │
│  affected. Then briefly summarize the rest of the list. Save the final report to data/report.md using the file  │
│  writer tool.                                                                                                   │
│  Agent: Product Insights Manager                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



===== HIERARCHICAL RESULT =====

# Product Feedback Stakeholder Report

This report summarizes recent product feedback, highlighting key priorities for the product team based on frequency of mentions and impact across customer tiers. The insights gathered will help inform our product roadmap and ensure we are addressing the most critical needs and opportunities.

## Top Priorities

1.  **Export Feature Issues:** This is the most frequently reported issue with 5 mentions, significantly impacting Enterprise and Pro users, as well as Free users. Addressing this bug is crucial for core functionality and user satisfaction.

2.  **Billing/Pricing/Support Issues:** With 4 mentions, these complaints are critical as they directly affect customer satisfaction and retention, particularly for our Enterprise clients, but also Pro and Free users.

3.  **Search Feature Issues:** This bug, mentioned 3 times, affects a fundamental aspect of the product and is reported across all customer tiers (Enter

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3e9972aa-e590-4a3a-8055-29b3eb319301                                                                       │
│  Final Output: # Product Feedback Stakeholder Report                                                            │
│                                                                                                                 │
│  This report summarizes recent product feedback, highlighting key priorities for the product team based on      │
│  frequency of mentions and impact across customer tiers. The insights gathered will help inform our product     │
│  roadmap and ensure we are addressing the most critical needs and opportunities.                                │
│                                                                                                                 │
│  ## Top Priorities                                                                                              │
│                                                                                                                 │
│  1.  **Export Feature Issues:** This is the most frequently reported issue with 5 mentions, significantly       │
│  impacting Enterprise and Pro users, as well as Free users. Addressing this bug is crucial for core             │
│  functionality and user satisfaction.                                                                           │
│                                                                                                                 │
│  2.  **Billing/Pricing/Support Issues:** With 4 mentions, these complaints are critical as they directly        │
│  affect customer satisfaction and retention, particularly for our Enterprise clients, but also Pro and Free     │
│  users.                                                                                                         │
│                                                                                                                 │
│  3.  **Search Feature Issues:** This bug, mentioned 3 times, affects a fundamental aspect of the product and    │
│  is reported across all customer tiers (Enterprise, Pro, and Free), indicating a broad impact on usability.     │
│                                                                                                                 │
│  ## Full Ranking                                                                                                │
│                                                                                                                 │
│  Below is a summary of the remaining feedback items:                                                            │
│                                                                                                                 │
│  *   **4. Slack/Teams Integration:** A highly requested feature (3 mentions) primarily from our Enterprise and  │
│  Pro users.                                                                                                     │
│  *   **5. Mobile App Logout Bug:** An important bug (2 mentions) affecting the mobile experience for            │
│  Enterprise and Pro users.                                                                                      │
│  *   **6. Higher API Rate Limits:** A specific feature request (2 mentions) from our Enterprise customers.      │
│  *   **7. Dark Mode:** A popular feature request (3 mentions) with interest from Pro and Free users.            │
│  *   **8. Speed Improvements:** Positive feedback (2 m

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Task 4: Hierarchical/Sequential Delegation - Findings

## Run Summary

The hierarchical crew was built by reusing the same three specialist agents
and tasks from Task 3, adding a manager agent (Product Insights Manager)
with `allow_delegation=True`, and running with `process=Process.hierarchical`
and `manager_agent=manager`.

## What Actually Happened

The run completed successfully with no errors. However, examining the
execution log shows that all three tasks were executed directly by the
manager agent itself:

appears as the executing agent for all three tasks, extraction,
prioritization, and report writing. The Feedback Aggregator, Prioritizer,
and Report Writer agents were never actually invoked. The manager read the
CSV, categorized the feedback, calculated priority scores, and wrote the
final report entirely on its own, rather than delegating each task to the
intended specialist.

This is a known behavior in CrewAI's hierarchical process, the manager
agent has the option to self-execute a task instead of delegating it, and
delegation is not guaranteed just because `allow_delegation=True` is set
and specialist agents are available.

## Token Usage Comparison

| Metric | Sequential | Hierarchical |
|---|---|---|
| Total tokens | 17,544 | 51,801 |
| Prompt tokens | 5,647 | 17,904 |
| Completion tokens | 11,897 | 33,897 |
| Successful requests | 6 | 17 |
| Who performed the work | 3 specialist agents, each on their own task | Manager agent alone, on all 3 tasks |

Hierarchical used roughly **3 times the tokens** of sequential for this run.

## Output Quality Comparison

Both runs produced well-formatted, complete reports following the same
Top Priorities plus Full Ranking structure. Despite the manager doing all
the work itself, output quality was comparable to the sequential run, both
correctly applied the priority scoring formula and produced a clear,
readable stakeholder report. Quality did not visibly improve from adding a
manager layer in this case, since no actual review or delegation between
distinct specialist perspectives took place.

## Reliability Comparison

Sequential ran cleanly and predictably through all three specialist
agents in order, matching the intended design exactly. Hierarchical also
completed without error, but did not execute as designed, the manager
bypassed delegation entirely, meaning the specialist roles, tools, and
personas built in Task 2 were never exercised in this run.

## Pros, Cons, When to Use

| | Pros | Cons | Use when |
|---|---|---|---|
| Sequential | Cheaper, faster, predictable order, specialist roles are guaranteed to run | No oversight or review layer if one agent's output is weak | Task order is fixed and each specialist agent is reliable on its own |
| Hierarchical | Can catch weak handoffs and add a review layer, in principle | Roughly 3x token cost, higher latency, and delegation to specialists is not guaranteed, the manager can self-execute and skip the crew entirely | Quality control matters more than cost, and only if delegation behavior is verified to actually occur |

## Key Finding

The most notable result from this comparison is not that hierarchical
produced better output, it did not, but that hierarchical delegation is
not reliable by default. In this run, the manager agent nearly tripled
token cost while bypassing the entire specialist crew, meaning none of
the benefits of role specialization (focused prompts, role-appropriate
tools, auditable per-stage output) were actually realized. This is an
important practical caveat for anyone considering `Process.hierarchical`
in production, the coordination overhead is paid regardless of whether
genuine delegation happens.


## Changing the backstory of the manager and re-checking whether it now delegates the tasks to the agents

## Note on Delegation-Fix Attempt

A follow-up run with an explicit anti-self-execution instruction in the 
manager's backstory was attempted. It did trigger genuine delegation 
attempts via the delegate_work_to_coworker tool, but repeatedly failed 
with a "coworker not found" error, where the delegation tool could not 
resolve the specialist agents by name despite them being correctly 
passed to the Crew.

**Root cause diagnosis:** the failure traces back to a subtlety in how 
CrewAI's Process.hierarchical interacts with task-level agent assignment. 
Sequential process requires agent= to be set directly on each Task, 
since there is no manager to decide ownership, the developer decides 
upfront in code. Hierarchical process assumes the opposite, the manager 
is meant to dynamically decide who does each task at runtime using its 
delegate_work_to_coworker tool, based on the roles of agents passed into 
Crew(agents=[...]). When a task already has agent= hardcoded, ownership 
is effectively decided twice, once statically in code and once 
dynamically by the manager, and these two sources of truth conflict. 
This produced the exact symptom observed: the delegation tool's internal 
coworker registry was built incorrectly and could only see the manager's 
own role.

This is a genuine subtlety rather than a simple misconfiguration, 
CrewAI does not raise an upfront validation error warning that agent= 
and Process.hierarchical should not be combined, tasks appear valid and 
run without complaint, the conflict only surfaces as inconsistent 
delegation behavior at runtime, discoverable only by reading the 
execution log closely rather than from any error message pointing 
directly at the cause.

A corrected version (tasks without agent= assigned) was not run to 
completion due to repeated gateway instability during testing, but the 
diagnosis itself is a valid and reproducible finding about CrewAI's 
hierarchical process that is not obvious from the documentation alone.

# Task 5: Evaluation & Cost Awareness

## Token Usage and Approximate Cost

| Run | Total Tokens | Successful Requests | Notes |
|---|---|---|---|
| Sequential | 17,544 | 6 | 3 specialist agents, one task each, ran as designed |
| Hierarchical (Run 1) | 51,801 | 17 | Manager self-executed all 3 tasks instead of delegating, roughly 3x the token cost of sequential |
| Single-agent LangGraph (Day 3) | Not logged | N/A | Token usage was not printed or saved during the Day 3 session, so no directly comparable figure is available. This is itself a useful takeaway, see note below. |

Approximate cost was not calculated in dollar terms since the gateway used
for this course is a shared/free-tier proxy rather than a metered API key
with a known per-token rate. Token count alone is used here as the
comparable cost proxy across runs, since all runs used the same
underlying model group where possible.

## Success Criteria

1. **Factual grounding:** Does the final report accurately reflect the
   actual feedback data, with correct mention counts and no invented
   issues?
2. **Completeness:** Does the report cover all major recurring issues
   from the dataset, not just the top one or two?
3. **Tone and clarity:** Is the report written in plain, confident
   business language a non-technical stakeholder could act on
   immediately?

## Manual Scoring (1 to 5 per criterion)

| Run | Factual Grounding | Completeness | Tone and Clarity |
|---|---|---|---|
| Sequential | 5 | 5 | 4 |
| Hierarchical (Run 1) | 5 | 5 | 4 |

Both runs scored identically on factual grounding, the priority
arithmetic was independently checked against the stated formula in both
runs and matched exactly. Both were also complete, covering all 13 to 19
distinct issues extracted from the dataset. Tone and clarity were both
solid business-appropriate prose, scored slightly below perfect only
because neither report added much beyond restating the ranked list in
sentence form, a genuinely sharper report might group related issues or
suggest concrete next steps rather than only restating rank order.

## Verdict

For this specific task and dataset size, the multi-agent crew was not
clearly worth its added complexity and cost. The sequential run produced
output of equal quality to the hierarchical run at roughly one third the
token cost, and the hierarchical run's manager agent did not even use the
specialist crew as designed, further undermining the case for the
extra coordination layer. Multi-agent specialization would likely become
worth the overhead on a larger, messier dataset where a single agent
might drop or blur categories, or where the prioritization formula was
complex enough to benefit from a dedicated, isolated reasoning step, but
at 30 rows of clean feedback, a single well-prompted agent following the
same explicit formula would very likely have matched this quality at
a fraction of the total token cost.

## Note on Missing Day 3 Comparison

 Single-agent LangGraph (Day 3) | Not precisely logged | N/A | max_tokens was capped at 1000 per call, so completion tokens per call were at most 1000, but actual usage was likely lower and prompt tokens were not tracked. No reliable total is available, only this upper bound on completion tokens per call.
